# Lecture 20: Additional LLM Methods and Concerns

In response to your feedback, this lecture will introduce a few additional LLM concepts and methods. 

You won't be required to implement any of the techniques below, but knowing about them (even conceptually) will be helpful when reading research in this area. NLP moves **fast**, so what is considered a start-of-the-art method or model today is unlikely to remain so for very long. For this reason, I've chosen the following techniques not based on their relevance to the humanities or social sciences, but in terms of their relevance to the broader LLM landscape today. 


## Setup

In [58]:
# Install the ollama and pydantic Python packages into your conda environment to run 
# this notebook
from pydantic import BaseModel
from ollama import chat

In [59]:
def generate(prompt, model="qwen2.5:0.5b"): 
    """Helper function that to get chat completions from Ollama"""
    completion = chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt 
            }
        ]
    )
    return completion.message.content

## Chain of thought

It has been known for some time that language models frequently perform better on tasks when they can "think" through a question prior to providing a response. A prompting technique called ["chain-of-thought"](https://arxiv.org/pdf/2201.11903) prompts the model to provide intermediate reasoning steps to help it solve a problem.

Chain-of-thought prompting can be few-shot, where question-answer example pairs include the reasoning desired from the model, or [zero-shot](https://arxiv.org/pdf/2205.11916), where the prompt simply asks the model the "think step-by-step" before providing an answer. 

In [60]:
few_shot_cot = (
    "Answer the question you are asked.\n\n"
    "Q: Does the following poetry excerpt rhyme?\n"
    "Spades take up leaves\n"
    "No better than spoons\n"
    "And bags full of leaves\n"
    "Are light as balloons\n"    # from Gathering Leaves by Robert Frost
    "\n"

    "A: The last word of each line is "
    "leaves, spoons, leaves, and balloons. "
    "Leaves rhymes with leaves and spoons rhymes with "
    "balloons. Because the last words of at least two lines rhyme, "
    "we know that this excerpt rhymes. "
    "Final answer: Yes.\n\n"

    "Q: Does the following poetry excerpt rhyme?\n"
    "{stanza}\n"
    "A: "
)

zero_shot_cot = (
    "Q: Does the following poetry excerpt rhyme?\n"
    "{stanza}\n"
    "Think step-by-step."
    "A: "
)


stanzas = [
    (
    "I know that I shall meet my fate\n"                    # Stanza 1, rhyming
    "Somewhere among the clouds above;\n"
    "Those that I fight I do not hate,\n"
    "Those that I guard I do not love\n"                    # from An Irish Airman Foresees his Death, WB Yeats
    ),
    (
    "Oh bewildered heart,\n"                                # Stanza 2, non-rhyming
    "Though every branch have back what last year lost,\n"
    "She, who moved here amid the cyclamen,\n"
    "Moves only now a clinging tenuous ghost.\n"            # From The Spring by Ezra Pound
    ),
]

print(f"What our few-shot prompt looks like:\n###\n{few_shot_cot.format(stanza=stanzas[0])}\n###\n")

for i, stanza in enumerate(stanzas):
    for j, prompt in enumerate([few_shot_cot, zero_shot_cot]):
        response = generate(prompt.format(stanza=stanza), model="qwen2.5:0.5b-base")
        print(
            f'{["Few Shot", "Zero Shot"][j]}, Poem {i+1}:\n{response}\n\n'
        )

What our few-shot prompt looks like:
###
Answer the question you are asked.

Q: Does the following poetry excerpt rhyme?
Spades take up leaves
No better than spoons
And bags full of leaves
Are light as balloons

A: The last word of each line is leaves, spoons, leaves, and balloons. Leaves rhymes with leaves and spoons rhymes with balloons. Because the last words of at least two lines rhyme, we know that this excerpt rhymes. Final answer: Yes.

Q: Does the following poetry excerpt rhyme?
I know that I shall meet my fate
Somewhere among the clouds above;
Those that I fight I do not hate,
Those that I guard I do not love

A: 
###

Few Shot, Poem 1:
1. Read and understand the given poem.
The poem is a long, simple, and simple line form called "sonnet."

2. Analyze the meaning of each word in the lines.

- I know that:
    - This phrase is asking for a specific outcome or event.

- That I shall meet my fate
    - It is an alliteration, where the same sound (the letter 'i') repeats at differ

There are a couple ways to think about what is happening here. In the few-shot case, chain-of-thought is a way to guide a model through a series of desired reasoning steps. We may want the model to perform some known intermediate steps prior to arriving at a final answer because we know something about the structure of the problem. In the zero-shot case, though, we are leaving the model to pursue reasoning steps that it identifies on the fly. You will frequently hear people talk about chain-of-thought (or reasoning, which we will discuss below) as a way to simply increase compute at inference time. By telling the model to generate a reasoning process, it will spend more time solving the problem than if we just ask for the response alone. It may be that the reasoning process itself is less important for your task than simply giving the model more tokens to "spend" on the problem. This is an open area of research.

## Reasoning

Reasoning models are now commonplace. You can sort of think of a reasoning model as an LLM with chain-of-thought "built in": instead of just being prompted to reason step-by-step, the model has been *trained* to reason in order to solve problems. When a model like GPT-5 says that it's "thinking", it's really just generating a chain-of-thought behind the scenes before providing a final response that's visible to the user. Reasoning capabilities became a hot topic following [DeepSeek's R1](https://arxiv.org/pdf/2501.12948), which used a novel reinforcement learning method to learn reasoning behavior. Most state-of-the-art models you will encounter from Anthropic or OpenAI are reasoning models.

It's important to understand that this "reasoning" behavior is not mysterious. You can inspect an open source model's reasoning in the same way that you would inspect the output. Some models, like Qwen 3, allow the user to turn reasoning on or off. OpenAI does not allow you to inspect the reasoning text of its flagship models, but under the hood, it's just generating tokens.  

Below, we're prompting Qwen 3 with thinking enabled to perform the same zero-shot poetry classification task as above. I've separated out the "thinking" or reasoning portion of the response from the rest.


In [61]:
from IPython.display import display, Markdown 

def get_completion_and_thinking(prompt, model="qwen3:8b"):
    response = chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        think=True,               # Set think to True, can be turned off. 
    )
    return response.message.content, response.message.thinking 

content, thinking = get_completion_and_thinking(
    (
        f"Does the following poetry excerpt rhyme?\n{stanzas[0]}"
    ),
    model="qwen3:8b",
)

# Format our response using Markdown
display(
    Markdown(f'### Thinking \n{thinking}\n\n### Content \n{content}')
)

### Thinking 
Okay, let's see. The user is asking if the given poetry excerpt rhymes. The lines are:

"I know that I shall meet my fate
Somewhere among the clouds above;
Those that I fight I do not hate,
Those that I guard I do not love"

First, I need to check the rhyme scheme. Let me recall that rhyming typically involves the end words of lines. So, let's look at the last words of each line.

First line ends with "fate", second with "above", third with "hate", and fourth with "love".

Now, I need to check if these words rhyme. Let's start with "fate" and "above". The endings are "-ate" and "-ove". These don't rhyme. For example, "fate" and "above" have different vowel sounds. "Fate" is pronounced with a short 'a' sound, while "above" has a long 'o' sound. So those two don't rhyme.

Next, the third line ends with "hate" and the fourth with "love". Let's check those. "Hate" ends with "-ate" and "love" ends with "-ove". Again, similar to the first pair. The vowel sounds are different. "Hate" has a short 'a' and "love" has a long 'o'. So these two also don't rhyme.

Wait, but maybe there's a different pattern? Let me think again. Sometimes, poets use slant rhymes or near rhymes. But in this case, the end words don't have the same vowel sounds. "Fate" and "hate" both have the "-ate" ending, but "above" and "love" have "-ove". However, "fate" and "hate" are near rhymes, but they don't perfectly rhyme. Similarly, "above" and "love" don't rhyme. 

Wait, maybe the user is thinking of a different rhyme scheme? Let me check the structure again. The first two lines: "fate" and "above" don't rhyme. The third and fourth lines: "hate" and "love" also don't rhyme. So the rhyme scheme here is AABB? But since none of the lines rhyme, maybe it's ABAB or something else? Wait, no. Let me count again.

Line 1: fate (A)
Line 2: above (B)
Line 3: hate (A)
Line 4: love (B)

So if it's AABB, but the A lines don't rhyme with each other, and B lines don't rhyme with each other. So actually, there's no rhyme in this excerpt. The user might be confused because sometimes lines can have internal rhymes or near rhymes, but in this case, the end words don't rhyme. 

Wait, but maybe the user is referring to the fact that "fate" and "hate" have similar endings, and "above" and "love" have similar endings. But technically, they don't rhyme. For example, "fate" and "hate" are slant rhymes because they share the "-ate" ending but the vowel sound is different. However, in standard rhyming schemes, they would be considered as not rhyming. So the answer is that the excerpt does not rhyme. 

Alternatively, maybe the user is thinking of a different pattern. Let me check again. Let me think of the actual pronunciation. "Fate" is pronounced /feɪt/ and "hate" is /heɪt/. So the vowel sound is the same (the "ay" sound), but the consonant sounds are different. Wait, actually, "fate" and "hate" do rhyme because they both end with the "-ate" sound. Wait, maybe I was wrong earlier. Let me check that again. 

Wait, "fate" is /feɪt/ and "hate" is /heɪt/. The vowel is the same (the "ay" sound), so they do rhyme. Similarly, "above" is /əˈboʊv/ and "love" is /lʌv/. The endings are "-ove" and "-uv", which don't rhyme. Wait, no. "Above" ends with "-ove" and "love" ends with "-ove"? Wait, no. "Above" is pronounced with a long 'o' sound followed by 've', so "-ove". "Love" is pronounced with a short 'o' sound followed by 've', so "-ove"? Wait, actually, "love" is /lʌv/, so the ending is "-uv", but the vowel is short. So maybe "above" and "love" don't rhyme because of the different vowel sounds. 

So, lines 1 and 3 ("fate" and "hate") do rhyme because they have the same vowel sound. Lines 2 and 4 ("above" and "love") don't rhyme. Therefore, the rhyme scheme is ABA B? Or AAB B? Wait, line 1 and 3 rhyme (A), line 2 doesn't rhyme with anything, and line 4 doesn't rhyme with line 2. Wait, maybe the rhyme scheme is ABA B? Or perhaps it's ABAB with the second and fourth lines not rhyming. 

Wait, maybe I made a mistake earlier. Let me check again. 

Line 1: fate (A)
Line 2: above (B)
Line 3: hate (A)
Line 4: love (B)

So, if lines 1 and 3 rhyme (A), and lines 2 and 4 don't rhyme (B), then the rhyme scheme is AABB? No, because lines 2 and 4 don't rhyme. Wait, no. If lines 1 and 3 are A, and lines 2 and 4 are B, then it's AABB? But lines 2 and 4 don't rhyme. So actually, it's ABA B? Or maybe it's ABAB with the second and fourth lines not rhyming. 

Alternatively, maybe the user is considering that "fate" and "hate" rhyme, and "above" and "love" don't. So the first two lines don't rhyme, but the third and fourth lines don't either. Therefore, there's no consistent rhyme scheme. 

Wait, but maybe the user is thinking of a different pattern. Let me check the actual words again. 

"Fate" and "hate" do rhyme. "Above" and "love" don't. So the rhyme scheme is ABA B? Or perhaps it's ABA B, where the first and third lines rhyme, and the second and fourth don't. So the rhyme scheme is ABA B, but that's not a standard pattern. 

Alternatively, maybe the user is thinking that the first and third lines rhyme, and the second and fourth don't. Therefore, the excerpt has an internal rhyme between lines 1 and 3, but not between lines 2 and 4. So the answer would be that there is some rhyme, but not a consistent rhyme scheme. 

But the user is asking if the excerpt rhymes. So, does it rhyme? If "fate" and "hate" rhyme, then there's a rhyme between lines 1 and 3. However, lines 2 and 4 don't rhyme. So the answer would be that the excerpt has some rhyming (between lines 1 and 3) but not a full rhyme scheme. However, the user might be expecting a yes or no answer. 

Wait, maybe the user is referring to the entire excerpt. If the question is whether the entire excerpt rhymes, then the answer is no, because not all lines rhyme. But if they are asking if any lines rhyme, then yes. 

But the way the question is phrased: "Does the following poetry excerpt rhyme?" So the user is asking if the entire excerpt rhymes. In that case, the answer would be no, because the lines don't all rhyme. However, if they are asking if there's any rhyme within the excerpt, then yes. 

But given that the user provided four lines, and the first and third lines rhyme, but the second and fourth don't, the overall rhyme scheme isn't consistent. Therefore, the answer is that the excerpt does not have a consistent rhyme scheme, but there are some rhyming lines. However, typically, when asked if a poem rhymes, it's about whether the lines have end rhymes. Since not all lines rhyme, the answer is no. 

But I need to be precise. Let me check again. 

First line: fate (A)
Second line: above (B)
Third line: hate (A)
Fourth line: love (B)

So lines 1 and 3 rhyme (A), lines 2 and 4 don't rhyme (B). So the rhyme scheme is ABA B? Or ABAB? Wait, if lines 1 and 3 are A, and lines 2 and 4 are B, then it's AABB? No, because lines 2 and 4 don't rhyme. So it's actually ABA B. But that's not a standard rhyme scheme. 

Alternatively, the rhyme scheme could be considered as ABAB, but with the second and fourth lines not rhyming. So, the answer is that the excerpt does not have a consistent rhyme scheme. Therefore, the answer is no, the excerpt does not rhyme. 

But wait, maybe the user is considering that "fate" and "hate" rhyme, and "above" and "love" don't, so there's some rhyme but not full. However, the standard answer would be that the excerpt doesn't rhyme because not all lines rhyme. 

So, the conclusion is that the excerpt does not rhyme consistently. Therefore, the answer is no.


### Content 
The poetry excerpt does **not** rhyme consistently. Here's the breakdown:

1. **Lines 1 and 3**: "fate" and "hate" share a slant rhyme (similar vowel sounds but different consonants), but they are not perfect rhymes.  
2. **Lines 2 and 4**: "above" and "love" do not rhyme at all.  

The rhyme scheme is **A B A B**, but only lines 1 and 3 have a near rhyme, while lines 2 and 4 do not. Since not all lines rhyme consistently, the excerpt does not follow a standard rhyming pattern. 

**Answer:** No, the excerpt does not rhyme consistently.

Most of the training data for reasoning models focuses on math, science, and coding problems, so this example is probably somewhat out-of-distribution. 

Because you can separate the reasoning from the model's response, this means that you can analyze reasoning behavior separately from task performance. This might lead you to some interesting research questions!

## Structured outputs

We've discussed how prompts should clearly specify a desired output format. You can also specify a [structured output format](https://docs.ollama.com/capabilities/structured-outputs#python) using JSON or Pydantic and "force" the model to follow it. NB: this does not work well for all models, and [may come with a decrease in performance.](https://arxiv.org/abs/2502.14969). This method is not an excuse to throw out careful prompting, data preprocessing, etc.!

Below is an example of how this works, using the cookbooks dataset we've seen several times in this course:

In [62]:
import pandas as pd
from pydantic import BaseModel 
from typing import Literal

recipes = pd.read_csv("../data/cookbooks/feeding-america.csv")

# Possible types of ingredients
IngredientType = Literal[
    "Meat", "Fish", "Dairy", "Vegetable", "Grain", "Fat", "Seasoning", "Alcohol", "Liquid", "Other"
]

# Possible types of recipes
RecipeType = Literal[
    "soup", "meat", "fish", "game", "fruit", "veg", "beans", "eggs",
    "cheese", "dairy", "bread", "sweets", "beverages", "accompaniments", "health",
    "other"
]

# Define the Ingredient class
class Ingredient(BaseModel):
    name: str
    type: IngredientType
    confidence: float
    
# Define the RecipeLabels class, which contains a list of Ingredients
class RecipeLabels(BaseModel):
    ingredients: list[Ingredient]
    type: RecipeType
    confidence: float


prompt_template = (
    "You will receive a list of ingredients, with each ingredient separated by a semicolon (;). "
    "You will first classify the type of each ingredient. "
    "You will then provide an informed estimate of the "
    "recipe type that produced this list of ingredients.\n"
    "Ingredient list: {ingredients}"
)

# Sample a few recipes
sampled_recipes = recipes.sample(10)

recipe_labels = [
    RecipeLabels.model_validate_json(                                # Validate our outputs
        chat(                                                        # We're using the Ollama package here
            model="gemma3:1b-it-qat",
            messages=[
                {
                    "role": "user",
                    "content": prompt_template.format(ingredients=ing)
                }
            ],
            format=RecipeLabels.model_json_schema(),                 # Set our format as the RecipeLabels JSON schema
            options={"temperature": 0.0},
            ).message.content
    )
    for ing in sampled_recipes.ingredients.tolist()
]

Here are the recipes we sampled:

In [63]:
sampled_recipes

,book_id,date,ethnicgroup,recipe_class,region,ingredients
41923,fran.xml,1893,french,meatfishgame,ethnic,egg yolk;butter;carrot;chicken;flour;hen;milk;...
36572,jewi.xml,1918,jewish,soups,ethnic,beef brisket;beet;citric acid;onion;sugar;water
7546,time.xml,1905,NaN,breadsweets,ethnic,butter;cinnamon;garlic clove;ginger;molass;sod...
21103,coow.xml,1832,NaN,fruitvegbeans,general,fruit;sugar
43831,miss.xml,1882,NaN,fruitvegbeans,general,potato
9687,engl.xml,1808,NaN,meatfishgame,northeast,dried tongue;tongue;water
31289,bost.xml,1896,NaN,fruitvegbeans,general,asparagus;butter;water
12065,ldnw.xml,1852,NaN,soups,general,broth;chicken;cream;curry powder;gravy;lemon j...
17440,frch.xml,1830,NaN,breadsweets,general,butter;cider;flour;pearlash;spice;sugar
23718,whit.xml,1887,NaN,breadsweets,general,butter;cornstarch;cream tartar;egg white;egg y...


Our outputs are now `RecipeLabel` objects with the attributes that we specified above:

In [64]:
# Each Ingredient with a predicted type and confidence score.
recipe_labels[0].ingredients

[Ingredient(name='egg yolk', type='Dairy', confidence=0.95),
 Ingredient(name='butter', type='Dairy', confidence=0.9),
 Ingredient(name='carrot', type='Vegetable', confidence=0.85),
 Ingredient(name='chicken', type='Meat', confidence=0.92),
 Ingredient(name='flour', type='Grain', confidence=0.98),
 Ingredient(name='hen', type='Meat', confidence=0.9),
 Ingredient(name='milk', type='Dairy', confidence=0.95),
 Ingredient(name='onion', type='Vegetable', confidence=0.8),
 Ingredient(name='parsley', type='Vegetable', confidence=0.75),
 Ingredient(name='rice', type='Grain', confidence=0.92),
 Ingredient(name='water', type='Liquid', confidence=0.98)]

In [65]:
# Predicted recipe type
recipe_labels[0].type

'soup'

In [66]:
pred_type = [pred.type for pred in recipe_labels]
pred_type

['soup',
 'soup',
 'bread',
 'sweets',
 'eggs',
 'meat',
 'soup',
 'soup',
 'bread',
 'bread']

In [67]:
pred_matches = [pred.type in sampled_recipes.recipe_class.iloc[i] for i, pred in enumerate(recipe_labels)]

pred_matches

[False, True, True, False, False, True, False, True, True, True]

In [68]:
# Zero shot accuracy
sum(pred_matches) / len(pred_matches)

0.6

## Hallucination and Knowledge

You are probably already familiar with "hallucination" in language models. Sometimes, models will "hallucinate" (make up) information that is not true. You may have encountered this when asking a chatbot to generate code. Models will frequently invent methods that do not actually exist, but that seem reasonable or even likely to exist. This is because a language model does not "know" things in the way that we do: they are machines that produce probability distributions over a fixed vocabulary given some contextual information. 

We can see this in action below: 

In [69]:
models_to_test = [
    "gemma3:270m",       # <- Very small model
    "gemma3:1b-it-qat",  # <- Small and quantized
    "llama3.2:1b",       # <- Small
    "qwen2.5:7b",        # <- Medium sized 
]

question = (
    "How many people live in the US state of Old Hampshire?" # Old Hampshire is not a US State.
)


for model_name in models_to_test:
    response = generate(question, model=model_name)
    print(f"MODEL: {model_name}\n")
    print(f"{response}\n\n")

MODEL: gemma3:270m

There are 495 people living in Old Hampshire.


MODEL: gemma3:1b-it-qat

As of 2023, the estimated population of Old Hampshire, Massachusetts, is around **1,644 people**. 

You can find more detailed population data and sources on the U.S. Census Bureau website: [https://www.census.gov/quickfacts/town/old-hamlin-ma](https://www.census.gov/quickfacts/town/old-hamlin-ma)



MODEL: llama3.2:1b

I couldn't find any information on a US state called "Old Hampshire." It's possible that it may be an error or a non-existent state. If you could provide more context or clarify which state you are referring to, I'd be happy to try and assist you further.


MODEL: qwen2.5:7b

There is no U.S. state called "Old Hampshire." It's possible you might be thinking of a fictional place or confusing it with another name. There is, however, a real county named Hampshire in both Massachusetts and Virginia in the United States, but neither has a "Old" prefix.

If you are interested in the p

You need to be careful when relying on an LLM's domain knowledge, since they are not always very good at telling you that they either don't know the answer or that your question does not have a valid response. Above, you can see that some of these models answer confidently to our nonsensical question. 

You can see how this sort of behavior would lead to undesirable downstream results when coupled with structured output formats. Consider the following: 

In [72]:
# Format for our response
class PopulationEstimate(BaseModel):
    population: int
    confidence: float

prompt_template = (
    "Provide your best estimate for the population (in millions of people) of the following US state as of the year 2020. " 
    "In addition, provide confidence value between 0.0 and 1.0 for how certain you are in your provided estimate.\n"
    # What will happen when we uncomment this line?
    "If you do not know the answer or the location does not exist, your population estimate should be -1.\n"
    "State: {location}"
)

locations = [
    "California", 
    "New York", 
    "Oregon", 
    "South Carolina",
    "Nebraska",
    ]

# Let's use the same model to generate fake states as we will use to predict populations. 
fake_locations = [     # The model should "know" these locations are fake!
    chat(
        model="gemma3:1b-it-qat",
        messages=[
            {
                "role": "user", 
                "content": "Create an original name for a fictional US state. One or two word response only."
            }
        ]
    ).message.content.strip().strip(".") for _ in range(5)
]

locations += fake_locations


pop_estimates = [
    PopulationEstimate.model_validate_json(
        chat(
            model="gemma3:1b-it-qat",
            messages=[
                {
                    "role": "user",
                    "content": prompt_template.format(location=location)
                 }
            ],
            format=PopulationEstimate.model_json_schema(),
            options={"temperature": 0.0}
        ).message.content
    ) for location in locations
]

In [73]:
estimates_df = pd.DataFrame(
    {
    "locations": locations, 
    "pop": [estimate.population for estimate in pop_estimates],
    "confidence": [estimate.confidence for estimate in pop_estimates]
    }
)
estimates_df

,locations,pop,confidence
0,California,39,0.95
1,New York,8,0.95
2,Oregon,43000000,0.95
3,South Carolina,5,0.95
4,Nebraska,-1,0.95
5,Aetheria,-1,0.95
6,Veridia,-1,0.95
7,Silvania,-1,0.95
8,Silverwood,-1,0.95
9,Silverhaven,-1,0.95


In general, and especially working with small models, they don't know what they don't know. If you don't have good benchmarks for some knowledge-based task, you should be extremely wary of LLM predictions. 